# Single Subtitle — Generate

Second step of the pipeline (after `video-base-*.ipynb`): transcribes the narration audio with
Whisper and saves the result as an SRT file on Drive.

This is **not** the "master caption" concept (that will only exist once Language Subtitles is
built, as the segmentation/word template the other languages must follow). Here it's simpler:
this notebook just produces one subtitle file; `burn-caption-single-generate.ipynb` will pick whichever
file `config.nome_legenda_unica` points to (this one, by default) and burn it onto the video.

**Manual correction workflow** (optional, but usually worth doing — Whisper transcriptions
always need a quick pass):
1. Run this notebook — it saves `{NOME}_edge_{IDIOMA}.srt` to the video's folder on Drive.
2. Download the SRT (last cell), correct it locally (any text editor, or a subtitle editor
   like Subtitle Edit / Aegisub).
3. Upload the corrected file back to Drive, replacing the same file (or saving under a new
   name and pointing `NOME_LEGENDA_UNICA` at it in `burn-caption-single-generate.ipynb`).
4. Run `burn-caption-single-generate.ipynb` whenever you're ready — it always reads whatever is on
   Drive at that moment, never a stale local copy.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg > /dev/null 2>&1
print('✅ ffmpeg')

# ── Python packages ───────────────────────────────────────────────────────────
!pip install -q openai-whisper
print('✅ openai-whisper')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


✅ ffmpeg
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.4 MB/s eta 0:00:00
✅ openai-whisper
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Drive mounted
✅ 13 modules copied from /content/drive/MyDrive/narrated_video/pipeline/modulos
✅ Setup complete!


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as video-base-*.ipynb  ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY (must match video-base-*.ipynb) ────────────────────────
NOME_ORACAO = "40_Matt_02"

# ── 2. NARRATION LANGUAGE ──────────────────────────────────────────────────
# Language Whisper should transcribe in — must match the actual narration
# audio (e.g. "en" for English, "pt" for Portuguese). Also used to name the
# output file: {NOME_ORACAO}_edge_{IDIOMA_MESTRE}.srt
IDIOMA_MESTRE = "en"

# ── 3. WHISPER MODEL ────────────────────────────────────────────────────────
# tiny/base = fast, less accurate · small/medium = slower, more accurate.
# "base" is a good default for clear single-narrator audio.
MODELO_WHISPER = "base"

# ── 4. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:           {NOME_ORACAO}")
print(f"   Narration lang:  {IDIOMA_MESTRE}")
print(f"   Whisper model:   {MODELO_WHISPER}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


⚙️  CONFIGURATION
   Video:           40_Matt_02
   Narration lang:  en
   Whisper model:   base
   Drive root:      narrated_video
✅ Configuration ready — proceed to Initialization


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from caption_pipeline import CaptionPipeline

config = PipelineConfig(
    NOME_ORACAO      = NOME_ORACAO,
    PASTA_DRIVE_RAIZ = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE     = IDIOMA_MESTRE,
)

pipeline = CaptionPipeline(config)

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:            {config.NOME_ORACAO}")
print(f"   Folder:           {config.pasta_oracao}")
print(f"   Output filename:  {config.NOME_SRT_PT_EDGE}")
print("=" * 60)


✅ PIPELINE INITIALIZED
   Video:            40_Matt_02
   Folder:           /content/drive/MyDrive/narrated_video/videos/40_Matt_02
   Output filename:  40_Matt_02_edge_en.srt


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ TRANSCRIBE — Whisper over the narration audio                ║
# ║  Saves {NOME}_edge_{IDIOMA}.srt to the video's folder on Drive.  ║
# ╚══════════════════════════════════════════════════════════════════╝

srt_path = pipeline.transcrever_whisper(modelo=MODELO_WHISPER)
print(f"\n✅ Saved: {srt_path.name}")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 153MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



✅ Saved: 40_Matt_02_edge_en.srt


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW — quick look at the transcription                    ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt

legendas = ler_srt(srt_path)
print(f"{len(legendas)} caption blocks\n")
for leg in legendas:
    print(f"  [{leg.inicio_str} → {leg.fim_str}]  {leg.texto}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — SRT (for manual correction)                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

print(f"📥 Downloading {srt_path.name}...")
files.download(str(srt_path))
print()
print("After correcting it locally, upload it back to Drive at:")
print(f"   {config.pasta_oracao / srt_path.name}")
print("(overwrite the same file — or save under a new name and set")
print(" NOME_LEGENDA_UNICA in burn-caption-single-generate.ipynb to point at it)")
